# Cloud9 scouting summary (Phase 5 check)

Reproduces a Cloud9 scouting summary from the Phase 5 statistics tables.
Run `make data` (or `uv run fele ingest && uv run fele static && uv run fele stats`) first.
Outputs are never committed (nbstripout), because they are derived from licensed GRID data.

Every rate is shown with its sample (`k`/`n`) and a 95% interval; tendencies are labelled strong / weak / noise.

In [ ]:
import polars as pl

from fele.config import settings

pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(60)
AGG = settings().local("aggregates")
TEAM = "Cloud9 Kia"
PERIOD = "recent"  # last 120 days; "all" for every game


def table(name: str) -> pl.DataFrame:
    return pl.read_parquet(AGG / f"stats_{name}.parquet")


teams = pl.read_csv(settings().meta_dir / "teams.csv", infer_schema=False).filter(pl.col("title") == "lol")
team_id = teams.filter(pl.col("name") == TEAM)["team_id"].item()
team_id

## Draft priorities (presence = picked or banned by Cloud9)

In [ ]:
draft = table("team_draft").filter(pl.col("team_id") == team_id, pl.col("period") == PERIOD)
draft.filter(pl.col("split_type") == "all").select(
    "champion", "picks", "bans", "k", "n", pl.col("rate", "lo", "hi", "base_rate").round(2),
    pl.col("first_rotation_rate").round(2), pl.col("avg_slot").round(1), "tendency",
).head(15)

### By game number in the series (Fearless: G1 / G2 / G3+)

In [ ]:
top = draft.filter(pl.col("split_type") == "all").head(8)["champion"]
draft.filter(pl.col("split_type") == "game_bucket", pl.col("champion").is_in(top.implode())).pivot(
    on="split", index="champion", values="rate"
)

## Ban targeting: which Cloud9 player do opponents' bans aim at?

In [ ]:
table("ban_targets").filter(pl.col("team_id") == team_id, pl.col("period") == PERIOD).select(
    "target_player", "target_role", "k", "n", pl.col("rate", "lo", "hi").round(2), "champions"
)

## Player pools (pro games)

In [ ]:
players = table("players").filter(pl.col("team_id") == team_id)
players.select(
    "player_name", "role", "n", "unique_champions",
    pl.col("effective_pool", "effective_pool_lo", "effective_pool_hi").round(1), "depth", "meta_k", "meta_n",
)

In [ ]:
pc = table("player_champions").join(players.select("player_id", "player_name"), on="player_id")
pc.sort("comfort", descending=True).group_by("player_name", maintain_order=True).head(5).select(
    "player_name", "champion", "k", "n", pl.col("shrunk", "shrunk_lo", "shrunk_hi", "recent_games", "comfort").round(2)
)

## Macro context

Positions are snapshots at event times and the feed has no normal camp events: "jungle top start" means the jungler
reached level 2 on the top half of the map.

In [ ]:
table("macro_rates").filter(pl.col("team_id") == team_id, pl.col("period") == PERIOD, pl.col("split") == "all").select(
    "metric", "k", "n", pl.col("rate", "lo", "hi", "base_rate").round(2), pl.col("p").round(3), "tendency"
)

In [ ]:
table("macro_timings").filter(pl.col("team_id") == team_id, pl.col("period") == PERIOD).with_columns(
    pl.col("median_s", "q25_s", "q75_s").cast(pl.Int64)
)

## Live meta and Cloud9's overlap

In [ ]:
meta = table("meta").filter(pl.col("window") == "live")
meta.select(
    "name", "k", "games", pl.col("presence", "pro_wr", "soloq_wr", "delta", "meta_score").round(3), "soloq_n"
).head(15)